This file was created with the assistance of Generative AI.  

# Intervention A — should `description` be embedded, and does its section structure matter?

The six-encoder selection left one prose field on TF-IDF. That was an inconsistent partition:
`title` and `blurb` were embedded because they are free text, while `description` — also free
text — stayed lexical, on the strength of a cost estimate and a guess about its content drawn from
six sampled rows. `creator` and `taxonomy` are genuinely categorical and correctly stay on exact
match; `description` never had a principled reason to.

This notebook tests three treatments of that field, holding **everything else fixed** — same ALS
fits, same `ref_train`, same `cold_val`, same warm guard, same lambda grid, same `title`+`blurb`
block:

| mode | `description` becomes |
|---|---|
| `tfidf` | one TF-IDF block, w=0.5 — what the selection ran |
| `pooled` | one embedding block, w=0.5 — section vectors averaged per item |
| `sections` | three embedding blocks (author_bio / editorial_review / jacket_copy), w=0.5/sqrt(3) each |

**No weight is tuned anywhere.** The section blocks split `reviews`' existing 0.5 between them, so
every field keeps the share `content.DEFAULT_WEIGHTS` gave it, total squared weight stays 3.5, and
prose stays 64.3% of an item's norm. The three rows differ only in how one field is represented.

**Why `sections` might beat `pooled`.** Averaging an author biography with a plot review produces a
vector that represents neither. Keeping them as separate blocks compares items bio-to-bio and
review-to-review, and `content.py`'s missing-field renormalization already handles the ~32-68%
coverage per block: an item with only a bio has that block's weight scaled up rather than being
penalised against an item with all three (gated in `bench_21`, section 3c).

In [8]:
%%time
import gc
import hashlib
import json
import os
import pickle
import sys
import time

sys.path.insert(0, "..")

import numpy as np
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

from recsys import load, cf, cbhcf, intervention_a as ia, eval as ev

MODEL = "arctic-l-v2.0"
DEVICE = "cuda:1"
DATA_PATH = "../data/filtered/books_5core_common.parquet"
SPLIT_PARAMS = dict(cold_item_fraction=0.10, cold_val_fraction=0.10)
EMBED_DIR = "../data/embeddings"

SELECTOR, K = "NDCG", 100
K_LEVELS_TUNE = [0, 2, 5, 10, 20]
N_SEEDS_TUNE = 2
ALS_PARAMS = dict(factors=64, regularization=0.01, iterations=20)
LAMBDA_GRID = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 16.0]
WARM_TOLERANCE, N_WARM_EVAL = 0.02, 10_000
VARIANTS = ["chunked", "pooled", "sections"]        # "tfidf" is read from the selection checkpoint, not re-run


def cache_pickle(name, compute, params=None):
    tag = "" if params is None else "_" + hashlib.md5(repr(params).encode()).hexdigest()[:8]
    path = f"../data/cache/{name}{tag}.pkl"
    if os.path.exists(path):
        with open(path, "rb") as f:
            return pickle.load(f)
    val = compute()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(val, f, protocol=pickle.HIGHEST_PROTOCOL)
    return val


dataset = cache_pickle("books_dataset", lambda: load.load_dataset(data_path=DATA_PATH, **SPLIT_PARAMS),
                       params=load.load_params_fingerprint(DATA_PATH, **SPLIT_PARAMS))
FP = load.dataset_fingerprint(dataset)
val = dataset.cold_val
print(f"cold_val items {len(val.cold_item_ids):,}   fingerprint {FP}")

<timed exec>:41: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.


cold_val items 2,727   fingerprint (384339, 487790, 2676601, 345998, 347561, 2727, 633483902, 13635, 2727, 625156242)
CPU times: user 74.4 ms, sys: 152 ms, total: 227 ms
Wall time: 226 ms


## Fixed apparatus

Identical to the selection notebook's, so every number here is comparable to the six-encoder table
and to TF-IDF's `cbhcf.coldval_at_selected_lambda` row in `hyperparams.json`.

In [9]:
%%time
als_seeds = cache_pickle(
    "als_seeds_tune",
    lambda: [cf.ALSModel(random_state=s, **ALS_PARAMS).fit(dataset.ref_train)
             for s in range(N_SEEDS_TUNE)],
    params=(FP, tuple(sorted(ALS_PARAMS.items())), N_SEEDS_TUNE))
for m in als_seeds:
    m.prepare_gpu_recommend(val, candidates="warm_cold", device=DEVICE)

warm_item_ids = np.unique(dataset.ref_train.nonzero()[1])
eval_users_val = np.flatnonzero(np.diff(val.test_matrix.tocsr().indptr))
_warm_pool = np.flatnonzero(np.diff(dataset.ref_val.tocsr().indptr))
warm_eval_users = np.sort(np.random.default_rng(0).choice(
    _warm_pool, min(N_WARM_EVAL, len(_warm_pool)), replace=False))
_keep = np.zeros(dataset.n_users, dtype=bool)
_keep[warm_eval_users] = True
ref_val_sample = dataset.ref_val.multiply(_keep[:, None]).tocsr()
ref_val_sample.eliminate_zeros()
cache_users = np.union1d(eval_users_val, warm_eval_users)
print(f"eval users {len(eval_users_val):,}   warm guard {len(warm_eval_users):,}   "
      f"cache covers {len(cache_users):,}")


def objective(by_k, ceiling, metric=SELECTOR):
    return float(np.nanmean(list(by_k[metric]) + [ceiling[metric]]))

<timed exec>:41: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.


eval users 12,264   warm guard 10,000   cache covers 21,899
CPU times: user 205 ms, sys: 2 s, total: 2.21 s
Wall time: 2.21 s


## The sweep

Per-variant checkpointing: this loop is ~1 h and each variant is independent, so an interruption
costs at most one.

In [10]:
%%time
CKPT = "../outputs/intervention_a_description_checkpoint.json"
CKPT_KEY = {"fingerprint": list(FP), "model": MODEL, "lambda_grid": LAMBDA_GRID,
            "seeds": N_SEEDS_TUNE, "k_levels": K_LEVELS_TUNE, "text_weight": ia.DEFAULT_TEXT_WEIGHT,
            "warm_tolerance": WARM_TOLERANCE, "groups": list(ia.DESCRIPTION_GROUPS)}
results = {}
if os.path.exists(CKPT):
    blob = json.load(open(CKPT))
    if blob.get("key") == CKPT_KEY:
        results = blob["results"]
        print(f"resuming: {len(results)} variant(s) done ({', '.join(results) or 'none'})")


def save_ckpt():
    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    json.dump({"key": CKPT_KEY, "results": results}, open(CKPT, "w"), default=float)


for mode in VARIANTS:
    if mode in results:
        print(f"\n{mode}: from checkpoint (obj {results[mode]['best_objective']:.5f})")
        continue
    t_mode = time.perf_counter()
    print(f"\n{'=' * 78}\ndescription_mode = {mode}\n{'=' * 78}", flush=True)

    space = ia.build_space(dataset, MODEL, catalogue="books", fit_rows=warm_item_ids,
                           embed_dir=EMBED_DIR, data_root="..", description_mode=mode, verbose=True)
    base = cbhcf.wrap_seeds(als_seeds, dataset.ref_train, space, cache_users, content_weight=1.0,
                            gpu_device=DEVICE, build_mode_b=False, cache_path=None)

    rows = []
    for lam in LAMBDA_GRID:
        t0 = time.perf_counter()
        ms = [m.with_content_weight(lam) for m in base]
        curve, _ = ev.sweep_mode_a_cached(ms, val, K_LEVELS_TUNE, K=K, verbose=False, with_auc=False)
        ceil = ev.ceiling_reference(ms, val, K=K)
        warm = ev.mode_a_metrics_at_k(ms[0], dataset.ref_train, ref_val_sample, K=K)
        by_k = {m: list(curve[m]["mean"]) for m in ev.METRICS}
        ceil_m = {m: ceil["mean"][m] for m in ev.METRICS}
        rows.append({"lambda": lam, "by_k": by_k, "ceiling": ceil_m,
                     "warm": warm[SELECTOR], "objective": objective(by_k, ceil_m)})
        print(f"  lambda={lam:<5g} obj {rows[-1]['objective']:.5f}  warm {warm[SELECTOR]:.5f}  "
              f"({time.perf_counter() - t0:.0f}s)", flush=True)

    scores = [r["objective"] for r in rows]
    warms = [r["warm"] for r in rows]
    floor = max(warms) * (1 - WARM_TOLERANCE)
    feasible = [i for i, w in enumerate(warms) if w >= floor]
    unconstrained = int(np.nanargmax(scores))
    best = max(feasible, key=lambda i: scores[i]) if feasible else unconstrained

    sel = [m.with_content_weight(rows[best]["lambda"]) for m in base]
    auc_curve, _ = ev.sweep_mode_a_cached(sel, val, K_LEVELS_TUNE, K=K, verbose=False, with_auc=True)
    auc = [float(v) for v in auc_curve["AUC"]["mean"]]

    results[mode] = {"best_lambda": rows[best]["lambda"], "best_objective": scores[best],
                     "best_warm": warms[best], "ndcg_by_k": rows[best]["by_k"]["NDCG"],
                     "ceiling_ndcg": rows[best]["ceiling"]["NDCG"], "auc_by_k": auc,
                     "constraint_was_binding": bool(best != unconstrained),
                     "objective_by_lambda": scores, "warm_by_lambda": warms,
                     "dense_dims": int(space.D.shape[1]), "minutes": (time.perf_counter() - t_mode) / 60}
    print(f"  -> lambda* {rows[best]['lambda']:g}  obj {scores[best]:.5f}  AUC@k0 {auc[0]:.4f}  "
          f"dense dims {space.D.shape[1]:,}  [{results[mode]['minutes']:.1f} min]", flush=True)
    save_ckpt()
    del base, space
    gc.collect()

resuming: 3 variant(s) done (chunked, pooled, sections)

chunked: from checkpoint (obj 0.03729)

pooled: from checkpoint (obj 0.03733)

sections: from checkpoint (obj 0.03714)
CPU times: user 1.48 ms, sys: 0 ns, total: 1.48 ms
Wall time: 952 μs


## Comparison

Against the two rows already on record, both measured on this exact harness: arctic with
`description` left on TF-IDF (the selected model), and the CBHCF TF-IDF baseline.

In [11]:
%%time
hp = json.load(open("../outputs/hyperparams.json"))
ref = {}
sel_ckpt = "../outputs/intervention_a_selection_checkpoint.json"
if os.path.exists(sel_ckpt):
    r = json.load(open(sel_ckpt))["results"].get(MODEL)
    if r:
        ref["arctic (description=tfidf)"] = {
            "best_lambda": r["best_lambda"], "best_objective": r["best_objective"],
            "best_warm": r["best_warm"], "auc_by_k": r["auc_by_k_at_best"]}
cv = hp.get("cbhcf", {}).get("coldval_at_selected_lambda")
if cv:
    ref["TF-IDF baseline (CBHCF)"] = {"best_lambda": cv["lambda"], "best_objective": cv["objective"],
                                      "best_warm": None, "auc_by_k": cv["auc_by_k"]}

print(f"{'variant':<30}{'lam*':>6}{'cold obj':>11}{'vs tfidf':>10}{'AUC@k0':>9}"
      f"{'warm NDCG':>11}{'dims':>7}")
rows_out = []
for name, r in list(ref.items()) + [(f"arctic (description={m})", results[m]) for m in VARIANTS
                                    if m in results]:
    base_obj = ref.get("arctic (description=tfidf)", {}).get("best_objective")
    delta = (r["best_objective"] / base_obj - 1) * 100 if base_obj else float("nan")
    warm = f"{r['best_warm']:.5f}" if r.get("best_warm") is not None else "     -"
    dims = r.get("dense_dims", "")
    print(f"{name:<30}{r['best_lambda']:>6g}{r['best_objective']:>11.5f}{delta:>9.1f}%"
          f"{r['auc_by_k'][0]:>9.4f}{warm:>11}{str(dims):>7}")

variant                         lam*   cold obj  vs tfidf   AUC@k0  warm NDCG   dims
arctic (description=tfidf)         3    0.03712      0.0%   0.7865    0.05073       
TF-IDF baseline (CBHCF)            8    0.04798     29.3%   0.7754          -       
arctic (description=chunked)       3    0.03729      0.4%   0.7861    0.05019   2048
arctic (description=pooled)        3    0.03733      0.6%   0.7853    0.05020   2048
arctic (description=sections)      3    0.03714      0.0%   0.7863    0.05039   4096
CPU times: user 2.22 ms, sys: 0 ns, total: 2.22 ms
Wall time: 1.79 ms


## Persist

In [12]:
%%time
hp["intervention_a_description"] = {
    "model": MODEL, "groups": {g: v for g, v in ia.DESCRIPTION_GROUPS.items()},
    "window_words": ia.WINDOW_WORDS, "window_overlap": ia.WINDOW_OVERLAP,
    "max_windows": ia.MAX_WINDOWS, "lambda_grid": LAMBDA_GRID, "seeds": N_SEEDS_TUNE,
    "variants": results, "dataset_fingerprint": list(FP),
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
json.dump(hp, open("../outputs/hyperparams.json", "w"), indent=2)
print("wrote ../outputs/hyperparams.json -> intervention_a_description")

wrote ../outputs/hyperparams.json -> intervention_a_description
CPU times: user 2.91 ms, sys: 0 ns, total: 2.91 ms
Wall time: 2.4 ms
